# Snæfellsjökull: cross-section, ice thickness and equilibrium line

This notebook draws a **vertical slice through the ice cap** along a line you choose, showing:

- the **surface** from the DEM (ice where there's glacier, rock elsewhere)
- an **estimate of the ice thickness**, and so of the rock bed underneath
- the terrain behind the line, drawn as the DEM's triangle mesh (like the wireframe in `../Web/Model/`)
- the **equilibrium line altitude (ELA)** as horizontal lines: one estimate for when the glacier was last in balance (1990s), and the snowline measured from this summer's satellite image

It needs no Earth Engine and no internet. It reads files the other notebooks have already written:

| File | Written by |
|---|---|
| `outputs/snaefellsjokull_dem_10m_isn93.tif` | `download_dem.ipynb` |
| `../Snæfellsjökull_outlines/outputs/snaefellsjokull_outlines_all_years.gpkg` | `download_outlines.ipynb` |
| `../Snæfellsjökull_imagery/outputs/true_color/*.tif` (optional, for the snowline) | `v1_satellite_clearest_scene.ipynb` |

## Rock or ice? What the DEM measures

The DEM is the **top surface**, whatever is there: the top of the ice on the glacier, bare rock everywhere else. It says nothing about what's underneath. A DEM made from satellite stereo images (like this one) sees the same thing your eye would.

So on the glacier there are two surfaces, and the DEM has only the upper one:

```
  ice surface   ← DEM (top of the glacier)
  ░░░ ice ░░░   ← thickness: not in the DEM
  rock bed      ← needs radar, or an estimate
```

**Measured thickness.** Snæfellsjökull's thickness was measured once, by radio-echo sounding (radar dragged across the ice) in 2003 (Davíðsdóttir, 2003, reported in Jóhannesson et al., 2011). The **average was only ~30 m**: 60–90 m in the main outlet tongues (Blágilsjökull, Jökulháls, Hyrningsjökull), more than 40–50 m just west and east of the summit, and large areas under 10–20 m. The gridded data aren't published.

**This notebook's estimate** (section 4) uses a standard rule of thumb from glacier physics: ice flows under its own weight, so a glacier is thick where its surface is gentle and thin where it's steep, and it thins to zero at its edge. The rule has one unknown, the ice's strength, which is set so that the average thickness matches the 30 m measured by radar. Treat the result as **±50%**: good for a picture of the shape, not for exact depths. For a modelled thickness map, see the global datasets of Farinotti et al. (2019) and Millan et al. (2022).

## The equilibrium line: a line, not a plane

Over a year a glacier **gains** mass from snowfall and **loses** mass from melt. High up, more snow falls than melts, so some of each winter's snow survives the summer (the **accumulation zone**). Low down, all the winter snow melts and then some of the old ice too (the **ablation zone**). The **equilibrium line** is the boundary where the two balance: gain = loss over the year. Its height is the **ELA**.

Your picture of a horizontal line that rises is a good first approximation, with four refinements:

1. **It's a line on the glacier surface, not a plane in the air.** In practice it's very close to the **snowline at the end of summer**: above it you see white snow from last winter, below it grey, older glacier ice. That's how it's measured from satellites (section 5).
2. **It moves every year, up and down.** A warm summer or a dry winter pushes it up; a cool summer or a snowy winter pulls it down. It's weather, year by year, around a long-term level set by climate.
3. **It isn't perfectly level.** Shady north slopes keep snow longer than sunny south slopes, and wind piles snow on the side away from the storms. On Snæfellsjökull it can differ by 100 m or more between sides (section 5 measures this).
4. **The long-term trend is up.** As summers warm, the average ELA rises. If it rises **above the summit**, there's no accumulation zone left: every summer melts the whole year's snow, the glacier only loses mass, and it will disappear. Snæfellsjökull's top is ~1,400 m, so it doesn't have much room.

**What the glacier does about it.** A glacier in balance has roughly 55–60% of its area above the ELA (the accumulation area ratio, AAR). When the ELA rises, the glacier is too big for its climate and shrinks from the bottom until that ratio is restored, unless the ELA is above the summit, in which case it never is.

## Outputs

Written to `outputs/cross_section/` next to this notebook. Each line in `TRANSECTS` gets its own folder:

| Path | Contents |
|---|---|
| `<line>/snaefellsjokull_cross_section.png` | The section with the DEM's own ice (`OUTLINE_YEAR`) and both ELA lines, with a locator map of estimated thickness |
| `<line>/snaefellsjokull_cross_section_1990s.png` | The same section with the 1999 ice and the 1990s ELA |
| `<line>/snaefellsjokull_cross_section_2026.png` | The same section with the 2025 ice (the newest outline) and the 2026 ELA |
| `<line>/snaefellsjokull_cross_section.csv` | The profile every 10 m: DEM surface, estimated rock bed, and ice thickness for each outline year drawn |
| `<line>/README.md` | Context for that line: where it runs, how much ice it lost, what the ELA lines and the thickness mean |
| `README.md` | Overview of all the lines |
| `snaefellsjokull_ela.csv` | Every ELA estimate made below (shared by all lines) |
| `snaefellsjokull_ice_thickness_estimate_10m_isn93.tif` | Estimated thickness (m) for the whole glacier, float32, on the DEM grid; no-data `-9999` outside the ice |

### Figure style

All figures use one minimal black-and-white style: white background, black text and lines, greys for secondary elements, no gridlines and no top or right axis borders. In the sections the ice is solid black; the locator maps show its estimated thickness in orange.

## 1. Setup

In [ ]:
# Install dependencies (run once)
%pip install -q rasterio geopandas matplotlib numpy pandas scipy pyproj

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from IPython.display import display
from matplotlib.collections import LineCollection
from matplotlib.colors import LightSource
from matplotlib.ticker import FuncFormatter
from pyproj import Transformer
from rasterio.features import geometry_mask
from rasterio.warp import Resampling, reproject
from scipy import ndimage
from shapely.geometry import LineString

In [ ]:
from cycler import cycler

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "text.color": "black",
    "axes.edgecolor": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "axes.prop_cycle": cycler(color=["black"]),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "axes.linewidth": 0.8,
    "axes.titlesize": 11,
    "axes.titlelocation": "left",
    "font.family": "sans-serif",
    "font.size": 10,
    "lines.linewidth": 1.2,
    "legend.frameon": False,
})

GREY = "#9a9a9a"
LIGHT_GREY = "#d4d4d4"
ICE = "black"          # ice fill in the sections
ICE_EDGE = "#555555"   # estimated bed line
COLUMN_COLOR = "#cfcfcf" # columns above the ice in the final figures
STACK_COLUMN_STYLE = {"color": "#9a9a9a", "alpha": 0.45}  # see-through columns in the stacked figure

## 2. Settings

Each line in `TRANSECTS` gets its own folder of outputs. A line is set by its two end points in longitude/latitude, drawn from the start (left of the figure) to the end (right). The figure looks across the line to the left of that direction, and the mesh shows the terrain on that side, so each line runs in the direction with DEM behind it.

- **`w_e_summit`**: west to east through the summit (Þúfur, 64.803° N, 23.779° W), the same direction as the profiles in Jóhannesson et al. (2011). Seen from the south.
- **`ssw_nne_summit`**: through the summit from the south tongue to the north-east lobe, where the ice has pulled back most among lines through the summit. Seen from the east-south-east.
- **`wnw_ese_north_flank`**: across the north flank, where most of the 1999 ice has gone. It misses the summit. Seen from the mountain side, looking north-north-east and downhill; its mesh comes from the wider DEM (`MESH_DEM_PATH`). To find coordinates, click a point in Google Maps or in the map in `../Snæfellsjökull_imagery/v0_satellite_annual_composites.ipynb`.

In [ ]:
# Cross-section lines: folder name → (start, end, description). Points are (longitude, latitude).
# The start is the left edge of the figure. The figure looks across the line to the left of start→end;
# pick the direction where the ground falls away behind the line, so the mesh fills the rock.
TRANSECTS = {
    "w_e_summit": ((-23.849, 64.803), (-23.700, 64.803),
                   "West to east through the summit"),
    "ssw_nne_summit": ((-23.8039, 64.7750), (-23.7544, 64.8310),
                       "South-south-west to north-north-east through the summit, across the south tongue and the north-east lobe"),
    "wnw_ese_north_flank": ((-23.8259, 64.8325), (-23.6864, 64.8039),
                            "West-north-west to east-south-east across the north flank"),
}
TRANSECT_STEP_M = 10

# 1 = true scale (a metre up is drawn as long as a metre across). Raise it to stretch the section vertically.
VERTICAL_EXAGGERATION = 1

# Rock is drawn as the DEM's triangle mesh behind the line (looking north), seen from the side
MESH_STEP_M = 40        # mesh spacing, as in the 3D viewer's wireframe
MESH_DEPTH_M = 4500     # how far behind the line the mesh reaches, at most (it stops at the DEM's edge)
MESH_COLOR = "#8a8a8a"
MESH_LINEWIDTH = 0.2

# Which year's outline counts as "glacier" in the section and the thickness estimate.
# 2014 matches the DEM best: both come from ArcticDEM, and the DEM's surface is a ~2008–2020 blend.
OUTLINE_YEAR = 2014

# Extra figures, one per period: the ice of that outline year (on the same rock bed) and that period's ELA line.
# 1999 is the outline closest to the 1990s; 2025 is the newest outline.
PERIODS = {"1990s": 1999, "2026": 2025}

# Ice thickness estimate (section 4)
TARGET_MEAN_THICKNESS_M = 30   # measured average, radio-echo sounding 2003
SLOPE_SMOOTH_M = 150           # smooth the surface over this distance before measuring slope
MIN_SLOPE_DEG = 3              # keeps the estimate finite on the flat summit dome
ICE_DENSITY = 900              # kg/m³
GRAVITY = 9.81                 # m/s²

# ELA estimates (section 5)
BALANCE_OUTLINE_YEAR = 1999    # the glacier was roughly in balance in the 1990s (Jóhannesson et al., 2011)
BALANCE_AAR = 0.58             # typical accumulation area ratio of a glacier in balance
SNOW_REFLECTANCE = 0.6         # visible reflectance above which a glacier pixel counts as snow, not ice
SNOWLINE_BAND_M = 10           # height step when searching for the snowline

DEM_PATH = Path("outputs/snaefellsjokull_dem_10m_isn93.tif")
# Wider copy of the DEM, for the mesh behind lines near the edge (download_dem.ipynb, section 10).
# If it's missing, the main DEM is used and the mesh stops at its edge.
MESH_DEM_PATH = Path("outputs/snaefellsjokull_dem_10m_isn93_wide.tif")
OUTLINES_PATH = Path("../Snæfellsjökull_outlines/outputs/snaefellsjokull_outlines_all_years.gpkg")
IMAGERY_DIR = Path("../Snæfellsjökull_imagery/outputs/true_color")

OUT_DIR = Path("outputs/cross_section")
OUT_DIR.mkdir(parents=True, exist_ok=True)
SLUG = "snaefellsjokull"
NODATA = -9999.0

## 3. Load the DEM and outlines

In [ ]:
with rasterio.open(DEM_PATH) as src:
    z = src.read(1, masked=True).filled(np.nan).astype("float64")
    transform, crs, PIXEL = src.transform, src.crs, src.res[0]
    profile = src.profile
    bounds = src.bounds

# The mesh behind each line comes from the wider DEM if there is one. Sea (no-data) counts as 0 m.
with rasterio.open(MESH_DEM_PATH if MESH_DEM_PATH.exists() else DEM_PATH) as src:
    z_mesh = np.nan_to_num(src.read(1, masked=True).filled(np.nan).astype("float64"), nan=0.0)
    mesh_transform, mesh_bounds = src.transform, src.bounds
print("Mesh from", MESH_DEM_PATH.name if MESH_DEM_PATH.exists() else DEM_PATH.name)

outlines = gpd.read_file(OUTLINES_PATH).to_crs(crs).sort_values("year").reset_index(drop=True)
print("Outline years:", outlines["year"].tolist())


def ice_mask(year):
    """Boolean grid: True inside that year's outline."""
    geom = outlines.loc[outlines["year"] == year, "geometry"]
    return ~geometry_mask(geom, out_shape=z.shape, transform=transform)


ice = ice_mask(OUTLINE_YEAR)
print(f"{OUTLINE_YEAR} outline: {ice.sum() * PIXEL**2 / 1e6:.2f} km² on the DEM grid")

## 4. Estimate the ice thickness

Glacier ice deforms once the stress on its bed reaches a limit (`τ`, the *basal shear stress*). For a glacier at that limit (the "perfect plasticity" approximation), two simple rules follow:

- **Where the surface slopes at angle α:** thickness `H = τ / (ρ g sin α)`. Steep surface → thin ice, gentle surface → thick ice.
- **Near the edge, at distance d from the margin:** `H = √(2 τ d / (ρ g))`, so ice thins to zero at the margin with a rounded profile.

The estimate takes the smaller of the two at each point. The surface is smoothed over `SLOPE_SMOOTH_M` first, because ice responds to the slope over several ice thicknesses, not to every bump. `τ` is then set so the average thickness matches the radar measurement, `TARGET_MEAN_THICKNESS_M`.

**Other years.** `τ` is set once, on the `OUTLINE_YEAR` (2014) ice, which matches the DEM. That also fixes the rock bed (the DEM minus the ice). For each period in `PERIODS`, the same rule with the same `τ` is applied to that year's outline, and the ice is put on top of the same rock. So between the figures the rock stays put and only the ice changes.

Caveats: the radar survey was in 2003, and the ice has thinned since (about 1.5 m per year in 1999–2008), but the thinnest edges have also melted away, so the average of what's left has changed less. The rule of thumb can't see crevasses, buried ridges or the crater under the summit. Treat thicknesses as ±50%.

In [ ]:
z_smooth = ndimage.gaussian_filter(np.nan_to_num(z, nan=np.nanmin(z)), sigma=SLOPE_SMOOTH_M / PIXEL)
dz_dy, dz_dx = np.gradient(z_smooth, PIXEL)
sin_slope = np.maximum(np.sin(np.arctan(np.hypot(dz_dx, dz_dy))), np.sin(np.radians(MIN_SLOPE_DEG)))


def thickness(tau, mask, dist_to_edge):
    """Perfect-plasticity thickness inside `mask`: the smaller of the slope rule and the edge rule."""
    rho_g = ICE_DENSITY * GRAVITY
    h = np.minimum(tau / (rho_g * sin_slope), np.sqrt(2 * tau * dist_to_edge / rho_g))
    return np.where(mask, h, 0.0)


# Find τ so the mean thickness over the OUTLINE_YEAR ice matches the target (thickness grows with τ, so bisect)
dist_to_edge = ndimage.distance_transform_edt(ice) * PIXEL
lo, hi = 1e3, 1e6
for _ in range(60):
    tau = (lo + hi) / 2
    lo, hi = (tau, hi) if thickness(tau, ice, dist_to_edge)[ice].mean() < TARGET_MEAN_THICKNESS_M else (lo, tau)
H = thickness(tau, ice, dist_to_edge)
bed = z - H  # the rock: the DEM minus the ice (equal to the DEM where there's no ice)


def ice_for(year):
    """Ice of another year's outline on the same rock bed and the same τ: (thickness grid, surface grid)."""
    mask = ice_mask(year)
    h = thickness(tau, mask, ndimage.distance_transform_edt(mask) * PIXEL)
    return h, bed + h


print(f"Basal shear stress τ = {tau / 1000:.0f} kPa (typical glaciers: 50–150 kPa)")
print(f"{OUTLINE_YEAR}: mean {H[ice].mean():.0f} m, 90th percentile {np.percentile(H[ice], 90):.0f} m, "
      f"max {H.max():.0f} m, volume ≈ {H.sum() * PIXEL**2 / 1e9:.2f} km³")

period_ice = {}
for period, year in PERIODS.items():
    h, surf = ice_for(year)
    period_ice[period] = {"year": year, "H": h, "surface": surf}
    on = h > 0
    print(f"{period} ({year} outline): mean {h[on].mean():.0f} m, max {h.max():.0f} m, "
          f"volume ≈ {h.sum() * PIXEL**2 / 1e9:.2f} km³")

with rasterio.open(OUT_DIR / f"{SLUG}_ice_thickness_estimate_10m_isn93.tif", "w",
                   **{**profile, "dtype": "float32", "nodata": NODATA}) as dst:
    dst.write(np.where(ice, H, NODATA).astype("float32"), 1)
    dst.set_band_description(1, "estimated ice thickness (m)")
    dst.update_tags(method=f"perfect plasticity, tau={tau:.0f} Pa, mean scaled to {TARGET_MEAN_THICKNESS_M} m",
                    outline_year=str(OUTLINE_YEAR))

## 5. Equilibrium line estimates

**No measured ELA exists for Snæfellsjökull.** A measured ELA needs a mass-balance programme: stakes drilled into the ice and snow pits dug every spring and autumn. In Iceland that is done only on the big ice caps (Vatnajökull, Hofsjökull, Langjökull) and a few others. Snæfellsjökull has only had its snout positions measured, since 1931. For comparison, the measured balanced-budget ELAs (the ELA at which each ice cap would neither grow nor shrink) are (Hannesdóttir et al., 2020):

| Ice cap | Balanced-budget ELA | Distance from Snæfellsjökull |
|---|---|---|
| Langjökull | 1,000–1,200 m | ~150 km east, inland |
| Hofsjökull | 1,100–1,300 m | ~230 km east, central highlands |
| Vatnajökull | ~1,000 m (south outlets) to 1,350 m (inland) | ~350 km east |
| Drangajökull | 600–700 m, the lowest in Iceland | ~150 km north, very maritime |

Snæfellsjökull sits at the sea in a wet, mild climate, so you'd expect something between Drangajökull and Langjökull. The two numbers below are this notebook's own. Both use the same method: **the ELA is the height with a certain share of that year's glacier above it** (the accumulation area ratio, AAR, read off the DEM). They differ in where the share comes from.

**a. 1990s: the share is assumed.** In the 1990s Snæfellsjökull was close to balance (its snouts barely moved). A glacier in balance has about `BALANCE_AAR` (58%) of its area above the ELA, so the ELA then is the elevation with 58% of the 1999 glacier above it. This uses today's DEM, which is up to tens of metres lower than the 1999 surface near the edges, so it's a slight underestimate.

**b. 2026: the share is measured.** At the end of summer, the part of the glacier still under snow is the part that gained mass that year, i.e. the accumulation area. On the newest true-colour image from `v1_satellite_clearest_scene.ipynb`, glacier pixels brighter than `SNOW_REFLECTANCE` count as snow and darker ones as bare ice. The share of the glacier (the outline closest to the image date) that is snow gives that year's AAR, and the ELA is the height with that share of the glacier above it. It's also done separately for the north, east, south and west faces, which shows how far from level the line is.

Why not just apply 58% to 2026 as well? 58% is the share for a glacier *in balance*. Applied to the 2025 glacier it gives the ELA that glacier would need to stop shrinking, not where the line actually was. That number is in the table below as a comparison, together with the snowline from the earlier method (the height that best splits snow above from ice below).

If the newest image is from early summer or right after a snowfall, the snow share will be too high and the ELA too low. Check the image.

In [ ]:
def elevation_above(mask, share):
    """Elevation with `share` of the masked area above it."""
    return float(np.percentile(z[mask & ~np.isnan(z)], 100 * (1 - share)))


def snowline_split(snow, glacier):
    """For comparison: the height that best splits the glacier into snow above and ice below
    (fewest snow pixels below it plus ice pixels above it). None if that's the very top."""
    heights, is_snow = z[glacier], snow[glacier]
    candidates = np.arange(np.floor(heights.min() / SNOWLINE_BAND_M) * SNOWLINE_BAND_M,
                           heights.max() + SNOWLINE_BAND_M, SNOWLINE_BAND_M)
    errors = [np.sum(~is_snow & (heights >= h)) + np.sum(is_snow & (heights < h)) for h in candidates]
    best = candidates[int(np.argmin(errors))]
    return None if best > heights.max() - SNOWLINE_BAND_M else float(best)


ela_rows = [{
    "kind": "balance", "period": "1990s", "year": BALANCE_OUTLINE_YEAR, "sector": "all",
    "estimate": f"Balance ELA: {BALANCE_AAR:.0%} of the {BALANCE_OUTLINE_YEAR} glacier above it (assumed)",
    "share_above": BALANCE_AAR,
    "ela_m": round(elevation_above(ice_mask(BALANCE_OUTLINE_YEAR), BALANCE_AAR)),
}]

# Newest true-colour scene with real blue band (Sentinel-2 or Landsat; MSS is excluded)
scenes = sorted(p for p in IMAGERY_DIR.glob(f"{SLUG}_*_*_*.tif") if "mss" not in p.name) if IMAGERY_DIR.exists() else []
snow_scene = None
if scenes:
    snow_scene = max(scenes, key=lambda p: p.stem.rsplit("_", 1)[-1])  # newest acquisition date
    scene_date = snow_scene.stem.rsplit("_", 1)[-1]
    scene_year = int(scene_date[:4])

    with rasterio.open(snow_scene) as src:
        rgb = np.full((3,) + z.shape, np.nan, "float32")
        reproject(src.read([1, 2, 3]).astype("float32"), rgb, src_transform=src.transform, src_crs=src.crs,
                  src_nodata=src.nodata, dst_transform=transform, dst_crs=crs, dst_nodata=np.nan,
                  resampling=Resampling.bilinear)
    visible = rgb.mean(axis=0) / 10000
    snow = visible > SNOW_REFLECTANCE

    # Outline closest in time to the image
    snow_outline_year = min(outlines["year"], key=lambda y: abs(y - scene_year))
    glacier = ice_mask(snow_outline_year) & ~np.isnan(visible)

    # Which way the surface faces: 0° = north, 90° = east, ...
    # (rows run south, so dz_dy is minus the northward slope; downhill = -gradient)
    aspect = np.degrees(np.arctan2(-dz_dx, dz_dy)) % 360
    sectors = {"north": (aspect >= 315) | (aspect < 45), "east": (aspect >= 45) & (aspect < 135),
               "south": (aspect >= 135) & (aspect < 225), "west": (aspect >= 225) & (aspect < 315)}

    for sector, faces in {"all": np.ones_like(glacier), **sectors}.items():
        part = glacier & faces
        share = float(snow[part].mean())  # measured share of the glacier still under snow
        ela_rows.append({
            "kind": "snowline", "period": str(scene_year), "year": scene_year, "sector": sector,
            "estimate": f"ELA from snow cover on {scene_date}: {share:.0%} of the {snow_outline_year} glacier "
                        f"above it (measured)",
            "share_above": round(share, 3),
            "ela_m": round(elevation_above(part, share)) if share >= 0.02 else None,
        })

    # Two more numbers for comparison, not drawn
    split = snowline_split(snow, glacier)
    ela_rows.append({
        "kind": "comparison", "period": str(scene_year), "year": scene_year, "sector": "all",
        "estimate": f"Snowline on {scene_date}, height that best splits snow above from ice below",
        "share_above": None, "ela_m": round(split) if split is not None else None,
    })
    ela_rows.append({
        "kind": "comparison", "period": str(scene_year), "year": snow_outline_year, "sector": "all",
        "estimate": f"ELA the {snow_outline_year} glacier would need to stop shrinking ({BALANCE_AAR:.0%} above it)",
        "share_above": BALANCE_AAR, "ela_m": round(elevation_above(ice_mask(snow_outline_year), BALANCE_AAR)),
    })
    print(f"Snow cover from {snow_scene.name}, glacier = {snow_outline_year} outline")
else:
    print("No true-colour GeoTIFF found; run ../Snæfellsjökull_imagery/v1_satellite_clearest_scene.ipynb for 2026.")

ela = pd.DataFrame(ela_rows)
ela.to_csv(OUT_DIR / f"{SLUG}_ela.csv", index=False)
ela

`share_above` is the share of the glacier above the line. A blank `ela_m` means almost no snow was left on that side (under 2%): the line was **above the ice**, and that side had no accumulation zone that year.

## 6. Building a cross-section

`build_transect` samples the DEM, the estimated rock bed and each year's ice along a line, builds the mesh of terrain behind it, and measures how much ice the line crossed in every mapped year.

In [ ]:
to_dem = Transformer.from_crs("EPSG:4326", crs, always_xy=True)
inv = ~transform


def room(x, y, nx_, ny_):
    """How far one can go from (x, y) in direction (nx_, ny_) and stay inside the mesh DEM."""
    limits = []
    for v, n, lo, hi in [(x, nx_, mesh_bounds.left + PIXEL, mesh_bounds.right - PIXEL),
                         (y, ny_, mesh_bounds.bottom + PIXEL, mesh_bounds.top - PIXEL)]:
        if n > 1e-9:
            limits.append((hi - v) / n)
        elif n < -1e-9:
            limits.append((lo - v) / n)
    return min(limits)


def compass(degrees):
    names = ["north", "north-north-east", "north-east", "east-north-east", "east", "east-south-east", "south-east",
             "south-south-east", "south", "south-south-west", "south-west", "west-south-west", "west",
             "west-north-west", "north-west", "north-north-west"]
    return names[int((degrees % 360) / 22.5 + 0.5) % 16]


def build_transect(start, end):
    """Everything one cross-section needs: profiles for each ice year, and the mesh behind the line."""
    x0, y0 = to_dem.transform(*start)
    x1, y1 = to_dem.transform(*end)
    length = np.hypot(x1 - x0, y1 - y0)
    dist = np.arange(0, length + TRANSECT_STEP_M / 2, TRANSECT_STEP_M)
    px, py = x0 + (x1 - x0) * dist / length, y0 + (y1 - y0) * dist / length

    # Fractional row/column of each point, for bilinear sampling of the grids
    cols, rows = inv * (px, py)
    coords = np.vstack([rows - 0.5, cols - 0.5])

    def sample(grid, order=1):
        return ndimage.map_coordinates(grid, coords, order=order, mode="nearest")

    inside = (cols >= 0) & (cols < z.shape[1]) & (rows >= 0) & (rows < z.shape[0])
    if not inside.all():
        print(f"  Warning: {np.sum(~inside)} points fall outside the DEM; shorten the line.")

    # Profiles: the DEM itself (OUTLINE_YEAR ice), and each period's ice on the same rock bed
    bed_profile = sample(bed)
    profiles = {}
    for name, year, h_grid, mask in [("combined", OUTLINE_YEAR, H, ice)] + \
            [(p, v["year"], v["H"], v["H"] > 0) for p, v in period_ice.items()]:
        on = sample(mask.astype(float), order=0) > 0.5
        th = np.where(on, sample(h_grid), 0.0)
        profiles[name] = {"year": year, "on_ice": on, "thick": th, "bed": bed_profile, "surface": bed_profile + th,
                          "H": h_grid, "mask": mask}
    # The combined figure uses the DEM surface exactly
    profiles["combined"]["surface"] = sample(z)
    profiles["combined"]["bed"] = profiles["combined"]["surface"] - profiles["combined"]["thick"]

    # The DEM mesh behind the line: a grid along it (u) and back from it (v), on the left of start→end,
    # i.e. what you see standing on the right-hand side of the line, looking across it
    ux, uy = (x1 - x0) / length, (y1 - y0) / length
    nx, ny = -uy, ux
    depth = min(MESH_DEPTH_M, room(x0, y0, nx, ny), room(x1, y1, nx, ny))
    mesh_u = np.arange(0, length + MESH_STEP_M / 2, MESH_STEP_M)
    mesh_v = np.arange(0, depth + 1e-6, MESH_STEP_M)
    U, V = np.meshgrid(mesh_u, mesh_v)
    mesh_cols, mesh_rows = ~mesh_transform * (x0 + ux * U + nx * V, y0 + uy * U + ny * V)
    mesh_z = ndimage.map_coordinates(z_mesh, [mesh_rows.ravel() - 0.5, mesh_cols.ravel() - 0.5],
                                     order=1, mode="nearest").reshape(U.shape)

    # Ice along the line in every mapped year (exact, from the outline polygons)
    line = LineString([(x0, y0), (x1, y1)])
    ice_km = {int(r.year): line.intersection(r.geometry).length / 1000 for r in outlines.itertuples()}

    section = pd.DataFrame({"distance_m": dist, "x_isn93": px.round(1), "y_isn93": py.round(1),
                            "dem_surface_m": profiles["combined"]["surface"].round(1),
                            "bed_estimate_m": bed_profile.round(1)})
    for prof in profiles.values():
        section[f"thickness_{prof['year']}_m"] = prof["thick"].round(1)

    return {"start": start, "end": end, "x0": x0, "y0": y0, "x1": x1, "y1": y1, "length": length, "dist": dist,
            "profiles": profiles, "mesh_u": mesh_u, "mesh_z": mesh_z, "mesh_depth": depth, "ice_km": ice_km,
            "section": section,
            "bearing": np.degrees(np.arctan2(x1 - x0, y1 - y0)) % 360,
            "looking": np.degrees(np.arctan2(nx, ny)) % 360}

## 7. The figures

`draw_section` draws one figure. Each line gets three, all on the same vertical range so they can be compared:

- **Combined:** the DEM's own ice (`OUTLINE_YEAR`, 2014) and both ELA lines.
- **1990s:** the 1999 ice and the 1990s ELA.
- **2026:** the 2025 ice (the newest outline) and the 2026 ELA.

Each has a locator map on top (estimated thickness in orange, that year's outline and the section line) and the section below, west on the left. Black is ice (estimated thickness). Dotted lines are the ELAs. The rock is the DEM's triangle mesh for the `MESH_DEPTH_M` behind the line, seen from the south and cut off at the rock surface. The mesh is the same in every figure: it's today's terrain.

The sections are drawn at true scale (`VERTICAL_EXAGGERATION = 1`), so the ice is a thin skin, as it really is: ~30 m on a 1,400 m mountain. Raise `VERTICAL_EXAGGERATION` to make it easier to see.

In [ ]:
balance = ela[ela["kind"] == "balance"].iloc[0]
current = ela[(ela["kind"] == "snowline") & (ela["sector"] == "all")]
current = current.iloc[0] if len(current) else None
shade = LightSource(azdeg=315, altdeg=45).hillshade(np.nan_to_num(z, nan=0), dx=PIXEL, dy=PIXEL)
extent = (bounds.left, bounds.right, bounds.bottom, bounds.top)


def plot_section(ax, tr, prof, lines, columns=False, column_style=None):
    """Draw one section into `ax`: the rock mesh, the ice, and the chosen ELA lines ('balance', 'current').

    `columns`: grey columns rise from every stretch of ice to the top of the panel.
    """
    km, mesh_u, mesh_z = tr["dist"] / 1000, tr["mesh_u"], tr["mesh_z"]
    y_bottom, y_top = tr["y_bottom"], tr["y_top"]
    on_ice, surface_line, bed_line = prof["on_ice"], prof["surface"], prof["bed"]
    column_style = column_style or {"color": COLUMN_COLOR}

    # --- Rock: the DEM mesh behind the line, seen from the side, shown only below the rock surface
    mkm = mesh_u / 1000
    mu = np.broadcast_to(mkm, mesh_z.shape)
    along = [np.column_stack([mkm, row]) for row in mesh_z]                                  # lines along the section
    across = np.stack([np.stack([mu[:-1], mesh_z[:-1]], -1), np.stack([mu[1:], mesh_z[1:]], -1)], -2)  # lines going back
    diagonal = np.stack([np.stack([mu[:-1, :-1], mesh_z[:-1, :-1]], -1),
                         np.stack([mu[1:, 1:], mesh_z[1:, 1:]], -1)], -2)                    # triangle diagonals
    rock_top = np.where(on_ice, bed_line, surface_line)
    rock = plt.Polygon(np.r_[[[km[0], y_bottom]], np.column_stack([km, rock_top]), [[km[-1], y_bottom]]],
                       closed=True, facecolor="none", edgecolor="none", zorder=1)  # only clips the mesh
    ax.add_patch(rock)
    mesh_lines = LineCollection(along + list(across.reshape(-1, 2, 2)) + list(diagonal.reshape(-1, 2, 2)),
                                colors=MESH_COLOR, linewidths=MESH_LINEWIDTH, zorder=1.5)
    mesh_lines.set_clip_path(rock)
    ax.add_collection(mesh_lines)

    # --- Grey columns from the ice up to the top edge
    if columns:
        ax.fill_between(km, surface_line, y_top, where=on_ice, linewidth=0, zorder=1.8, **column_style)

    # --- Ice on top, and the section line
    ax.fill_between(km, bed_line, surface_line, where=on_ice, color=ICE, linewidth=0, zorder=2)
    ax.plot(km, np.where(on_ice, bed_line, np.nan), color=ICE_EDGE, linewidth=0.9, linestyle=(0, (3, 2)), zorder=2.1)
    ax.plot(km, surface_line, color="black", linewidth=1.2, zorder=2.2)

    # ELA lines, both dotted and labelled at the right
    ELA_LABEL_SIZE = 11
    ela_style = dict(color="black", linewidth=1, linestyle=(0, (1, 2)), zorder=3)
    if "balance" in lines:
        ax.axhline(balance["ela_m"], **ela_style)
        ax.text(km[-1], balance["ela_m"] + 8, "1990 ELA ", ha="right", va="bottom", fontsize=ELA_LABEL_SIZE)
    if "current" in lines and current is not None:
        if pd.notna(current["ela_m"]):
            ax.axhline(current["ela_m"], **ela_style)
            ax.text(km[-1], current["ela_m"] + 8, f"{current['year']} ELA ", ha="right", va="bottom",
                    fontsize=ELA_LABEL_SIZE)
        else:
            ax.text(km[-1], np.nanmax(surface_line) + 20, f"{current['year']}: no snow survived the summer ",
                    ha="right", va="bottom", fontsize=8)

    ax.set_xlim(km[0], km[-1])
    ax.set_ylim(y_bottom, y_top)
    ax.set_xticks([])
    ax.set_yticks([t for t in ax.get_yticks() if y_bottom <= t <= y_top - 60])  # keep labels off the top edge
    ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:,.0f} m"))
    ax.tick_params(axis="y", labelsize=6.5)
    ax.spines[["left", "bottom"]].set_visible(False)  # tick marks only, no axis lines


def draw_section(tr, prof, lines, out_path, show_map=True, columns=False, dpi=200):
    """One figure: locator map, section with that ice, and the chosen ELA lines ('balance', 'current').

    With `show_map=False` there is no locator map and no margin above the section, so the top of the
    section is the top edge of the image. With `columns=True`, a grey column rises from every stretch of
    ice to the top of the section.
    """
    x0, y0, x1, y1, length = tr["x0"], tr["y0"], tr["x1"], tr["y1"], tr["length"]
    km, mesh_u, mesh_z = tr["dist"] / 1000, tr["mesh_u"], tr["mesh_z"]
    y_bottom, y_top, thickness_max = tr["y_bottom"], tr["y_top"], tr["thickness_max"]
    on_ice, surface_line, bed_line = prof["on_ice"], prof["surface"], prof["bed"]

    # Layout in inches, top to bottom: locator map, section
    FIG_W, LEFT, RIGHT = 13.0, 0.9, 0.3
    ax_w = FIG_W - LEFT - RIGHT
    section_h = ax_w * (y_top - y_bottom) / length * VERTICAL_EXAGGERATION
    top_h, map_h, gap_map, bottom_h = (0.1, 1.7, 0.3, 0.1) if show_map else (0, 0, 0, 0.1)
    fig_h = top_h + map_h + gap_map + section_h + bottom_h
    fig = plt.figure(figsize=(FIG_W, fig_h))

    def axes_at(left, top, width, height):
        """Axes placed in inches from the figure's top-left corner."""
        return fig.add_axes([left / FIG_W, 1 - (top + height) / fig_h, width / FIG_W, height / fig_h])

    ax = axes_at(LEFT, top_h + map_h + gap_map, ax_w, section_h)

    if show_map:
        ax_map = axes_at(LEFT, top_h, 3.0, map_h)

        # --- Locator map: hillshade + estimated thickness, the outline and the section line
        ax_map.imshow(shade, cmap="gray", extent=extent, vmin=-0.3, vmax=1.3)
        thick_img = ax_map.imshow(np.where(prof["mask"], prof["H"], np.nan), cmap="Oranges", extent=extent,
                                  vmin=0, vmax=thickness_max)
        outlines.loc[outlines["year"] == prof["year"]].boundary.plot(ax=ax_map, color="black", linewidth=0.6)
        ax_map.plot([x0, x1], [y0, y1], color="black", linewidth=1.4)
        gx0, gy0, gx1, gy1 = outlines.loc[outlines["year"] == prof["year"]].total_bounds
        pad = 500
        ax_map.set_xlim(min(x0, x1, gx0) - pad, max(x0, x1, gx1) + pad)
        ax_map.set_ylim(min(y0, y1, gy0) - pad, max(y0, y1, gy1) + pad)
        ax_map.set_aspect("equal")
        ax_map.set_anchor("W")
        ax_map.axis("off")

        # Colorbar beside the map, wherever the equal aspect left it
        ax_map.apply_aspect()
        map_box = ax_map.get_position()
        cax = fig.add_axes([map_box.x1 + 0.006, map_box.y0, 0.005, map_box.height])
        cbar = fig.colorbar(thick_img, cax=cax)
        cbar.set_label("Estimated ice thickness (m)", fontsize=7, labelpad=2)
        cbar.ax.tick_params(labelsize=7, length=2)
        cbar.outline.set_visible(False)

    plot_section(ax, tr, prof, lines, columns=columns)

    if show_map:
        fig.savefig(out_path, dpi=dpi, bbox_inches="tight", pad_inches=0.25)
    else:
        fig.savefig(out_path, dpi=dpi)  # no trimming or padding: the section's top is the image's top
    plt.show()


def draw_stacked(tr, below, above, lines_below, lines_above, out_path, dpi=200):
    """Two sections of one line stacked: `above` directly on top of `below`, no map, no margins.

    Grey columns rise from the ice in both panels to the top edge of the image; the lower panel's columns
    carry on straight through the upper panel. Columns are see-through, so where both overlap they're darker.
    """
    FIG_W, LEFT, RIGHT = 13.0, 0.9, 0.3
    ax_w = FIG_W - LEFT - RIGHT
    panel_h = ax_w * (tr["y_top"] - tr["y_bottom"]) / tr["length"] * VERTICAL_EXAGGERATION
    panel_h = round(panel_h * dpi) / dpi  # whole pixels, so the two panels meet exactly on a pixel row
    bottom_h = round(0.1 * dpi) / dpi
    fig_h = 2 * panel_h + bottom_h
    fig = plt.figure(figsize=(FIG_W, fig_h))
    ax_top = fig.add_axes([LEFT / FIG_W, (bottom_h + panel_h) / fig_h, ax_w / FIG_W, panel_h / fig_h])
    ax_bottom = fig.add_axes([LEFT / FIG_W, bottom_h / fig_h, ax_w / FIG_W, panel_h / fig_h])
    for ax_ in (ax_top, ax_bottom):
        ax_.patch.set_alpha(0)

    # The lower panel's columns: one shape per stretch of ice, from that ice to the top edge of the image, on a
    # layer behind both panels (same scale as the lower panel, twice as tall), so it shows through the upper one
    ax_columns = fig.add_axes([LEFT / FIG_W, bottom_h / fig_h, ax_w / FIG_W, 2 * panel_h / fig_h], zorder=-1)
    km = tr["dist"] / 1000
    column_top = tr["y_bottom"] + 2 * (tr["y_top"] - tr["y_bottom"])
    ax_columns.fill_between(km, below["surface"], column_top, where=below["on_ice"], linewidth=0,
                            **STACK_COLUMN_STYLE)
    ax_columns.set_xlim(km[0], km[-1])
    ax_columns.set_ylim(tr["y_bottom"], column_top)
    ax_columns.axis("off")

    plot_section(ax_bottom, tr, below, lines_below)
    plot_section(ax_top, tr, above, lines_above, columns=True, column_style=STACK_COLUMN_STYLE)
    fig.savefig(out_path, dpi=dpi)  # no trimming: the top panel's top is the image's top
    plt.show()

## 8. Draw every line

One folder per line in `TRANSECTS`, each with the three figures and the profile CSV.

In [ ]:
FIGURES = [  # (file suffix, profile, ELA lines drawn)
    ("", "combined", ["balance", "current"]),
    ("_1990s", "1990s", ["balance"]),
    ("_2026", "2026", ["current"]),
]
sources = dict(zip(outlines["year"].astype(int), outlines["source"]))

transects = {}
for name, (start, end, description) in TRANSECTS.items():
    print(f"\n{name}: {description}")
    tr = build_transect(start, end)
    profiles = tr["profiles"]
    # Same vertical range and thickness scale for the three figures of one line, so they can be compared
    all_beds = np.concatenate([p["bed"] for p in profiles.values()])
    tr["y_bottom"] = np.floor(np.nanmin(np.r_[all_beds, tr["mesh_z"].ravel()]) / 25) * 25
    tr["y_top"] = np.ceil((max(np.nanmax(p["surface"]) for p in profiles.values()) + 250) / 100) * 100
    tr["thickness_max"] = np.ceil(max(np.percentile(p["H"][p["mask"]], 99) for p in profiles.values()) / 10) * 10
    transects[name] = tr

    folder = OUT_DIR / name
    folder.mkdir(parents=True, exist_ok=True)
    tr["section"].to_csv(folder / f"{SLUG}_cross_section.csv", index=False)
    for suffix, key, lines in FIGURES:
        draw_section(tr, profiles[key], lines, folder / f"{SLUG}_cross_section{suffix}.png")

    a, b = tr["ice_km"][BALANCE_OUTLINE_YEAR], tr["ice_km"][PERIODS["2026"]]
    print(f"  {tr['length'] / 1000:.1f} km long, seen looking {compass(tr['looking'])}; mesh {tr['mesh_depth'] / 1000:.1f} km deep")
    print(f"  ice along the line: {a:.2f} km in {BALANCE_OUTLINE_YEAR} → {b:.2f} km in {PERIODS['2026']} "
          f"({(b - a) / a:+.0%})")

## 9. READMEs

A `README.md` in each folder explains that line: where it runs, how much ice it lost (from the outline polygons, exactly), the two ELA lines and the thickness estimate. A `README.md` in `outputs/cross_section/` lists all the lines.

In [ ]:
import re


def fmt_point(lon_lat):
    lon, lat = lon_lat
    return f"{lat:.4f}° N, {abs(lon):.4f}° W"


def transect_readme(name, tr, description):
    a, b = tr["ice_km"][BALANCE_OUTLINE_YEAR], tr["ice_km"][PERIODS["2026"]]
    first = min(tr["ice_km"])
    look = compass(tr["looking"])
    snow_date = re.search(r"\d{4}-\d{2}-\d{2}", current["estimate"]).group() if current is not None else ""
    left_end, right_end = compass(tr["bearing"] + 180), compass(tr["bearing"])
    ice_rows = "\n".join(
        f"| {'~1890' if y == 1890 else y} | {km_:.2f} | {sources.get(y, '')} |" for y, km_ in sorted(tr["ice_km"].items()))
    thick_rows = "\n".join(
        f"| {key} | {p['year']} | {p['on_ice'].sum() * TRANSECT_STEP_M / 1000:.2f} | {p['thick'].max():.0f} |"
        for key, p in tr["profiles"].items())
    snow_line = (f"**2026 ELA ≈ {current['ela_m']:,.0f} m.** The height with {current['share_above']:.0%} of the "
                 f"{PERIODS['2026']} glacier above it. That share is *measured*: it's the part of the glacier still "
                 f"under snow at the end of summer, on the Sentinel-2 image of {snow_date}. "
                 f"Snow that survives the summer is the part of the glacier that gained ice that year."
                 if current is not None and pd.notna(current["ela_m"]) else "**2026 ELA:** not available.")
    return f"""# Snæfellsjökull cross-section: {description.lower()}

![Cross-section, combined]({SLUG}_cross_section.png)

## The line

- From **{fmt_point(tr['start'])}** (left edge of the figures, the {left_end} end) to **{fmt_point(tr['end'])}** (right edge, the {right_end} end).
- {tr['length'] / 1000:.1f} km long. The figures look **{look}** across the line: the mesh is the terrain up to {tr['mesh_depth'] / 1000:.1f} km behind it, in that direction.
- Drawn at true scale: a metre up is as long as a metre across.

## How much the ice has shrunk along this line

Between {BALANCE_OUTLINE_YEAR} and {PERIODS['2026']}, the ice along this line went from **{a:.2f} km to {b:.2f} km ({(b - a) / a:+.0%})**. Since ~{first} it went from {tr['ice_km'][first]:.2f} km ({(b - tr['ice_km'][first]) / tr['ice_km'][first]:+.0%}).

| Outline year | Ice along the line (km) | Mapped from |
|---|---|---|
{ice_rows}

Outlines: Veðurstofa Íslands (Icelandic Met Office), CC BY 4.0. The ~1890 outline is reconstructed from the moraines the glacier left at its Little Ice Age maximum.

## Files

| File | Ice drawn | ELA lines |
|---|---|---|
| `{SLUG}_cross_section.png` | {OUTLINE_YEAR} (the year that matches the DEM's surface) | 1990 and 2026 |
| `{SLUG}_cross_section_1990s.png` | {PERIODS['1990s']} (the outline closest to the 1990s) | 1990 |
| `{SLUG}_cross_section_2026.png` | {PERIODS['2026']} (the newest outline) | 2026 |
| `{SLUG}_cross_section.csv` | The profile every {TRANSECT_STEP_M} m: DEM surface, estimated rock bed, and ice thickness for each year drawn | |

The three figures share one vertical range and one thickness colour scale, so they can be laid side by side. The rock bed and the mesh are the same in all three; only the ice changes.

| Figure | Ice year | Ice along the line (km, on the DEM grid) | Thickest (m, estimate) |
|---|---|---|---|
{thick_rows}

## The two dotted lines

Both are the **equilibrium line altitude (ELA)**: the height where a year's snowfall and melt cancel out. Above it the glacier gains ice, below it, it loses ice. Both are worked out the same way, as *the height with a certain share of that year's glacier above it*, read off the DEM. They differ in where the share comes from.

- **1990 ELA ≈ {balance['ela_m']:,.0f} m.** The height with {balance['share_above']:.0%} of the {BALANCE_OUTLINE_YEAR} glacier above it. That share is *assumed*: about 58% of a glacier's area lies above its ELA when it is in balance (neither growing nor shrinking), and Snæfellsjökull was close to balance in the 1990s.
- {snow_line}

The line rose by about {current['ela_m'] - balance['ela_m']:,.0f} m, and the share of the glacier above it fell from {balance['share_above']:.0%} to {current['share_above']:.0%}. That is why the glacier keeps shrinking: most of it now sits where it loses ice every summer. If the line rises above the summit (~1,440 m), no snow can survive anywhere and the ice cap will disappear.

No *measured* ELA exists for Snæfellsjökull (it has no mass-balance stakes). For comparison, the measured balance ELA of Langjökull, ~150 km east, is 1,000–1,200 m.

## How the ice thickness is estimated

Snæfellsjökull's thickness was measured once, by radar in 2003: ~30 m on average, 60–90 m in the outlet tongues. No map of the bed was published, so the thickness here is an estimate from a standard rule of glacier physics ("perfect plasticity"): ice flows under its own weight, so it is thin where the surface is steep, thick where it is gentle, and thins to zero at the edge. The one unknown (the stress at which ice flows, τ ≈ {tau / 1000:.0f} kPa) is set so the average thickness of the {OUTLINE_YEAR} glacier is 30 m. Treat thicknesses as ±50%.

The rock bed is the DEM minus the {OUTLINE_YEAR} ice. The 1990s and 2026 figures put that year's ice, estimated the same way, on the same bed, so they show where the ice was, not the exact height of its surface then.

## Sources

- Surface: ÍslandsDEM v1.0 (10 m), Landmælingar Íslands, built from ArcticDEM; a blend of ~2008–2020 surfaces.
- Outlines: Veðurstofa Íslands (Icelandic Met Office) national glacier inventory, CC BY 4.0.
- Snow cover: Copernicus Sentinel-2.
- Radar thickness and the 1990s balance: Jóhannesson et al. (2011), *Jökull* 61. Langjökull ELA: Hannesdóttir et al. (2020), *Jökull* 70.

Made by `../../cross_section.ipynb`.
"""


for name, tr in transects.items():
    (OUT_DIR / name / "README.md").write_text(transect_readme(name, tr, TRANSECTS[name][2]), encoding="utf-8")

# Overview of all lines, in the parent folder
rows = []
for name, tr in transects.items():
    a, b = tr["ice_km"][BALANCE_OUTLINE_YEAR], tr["ice_km"][PERIODS["2026"]]
    rows.append(f"| [`{name}/`]({name}/README.md) | {TRANSECTS[name][2]} | {tr['length'] / 1000:.1f} km | "
                f"{a:.2f} → {b:.2f} km ({(b - a) / a:+.0%}) |")
overview = f"""# Snæfellsjökull cross-sections

Vertical slices through the ice cap, made by `../cross_section.ipynb`. Each folder has the same three figures (combined, 1990s, 2026), the profile as a CSV, and a README explaining the line, the ice, the ELA lines and the thickness estimate.

| Folder | Line | Length | Ice along the line, {BALANCE_OUTLINE_YEAR} → {PERIODS['2026']} |
|---|---|---|---|
{chr(10).join(rows)}

Shared by all lines:

| File | Contents |
|---|---|
| `{SLUG}_ela.csv` | Every ELA estimate: the 1990s and 2026 lines, by side of the mountain, and two comparison values |
| `{SLUG}_ice_thickness_estimate_10m_isn93.tif` | Estimated {OUTLINE_YEAR} ice thickness (m) on the DEM grid; no-data `-9999` outside the ice |
"""
(OUT_DIR / "README.md").write_text(overview, encoding="utf-8")
print(overview)

## References

- Jóhannesson, T., Björnsson, H., Pálsson, F., Sigurðsson, O. and Þorsteinsson, Þ. (2011). [LiDAR mapping of the Snæfellsjökull ice cap, western Iceland](https://www.vedur.is/media/vedurstofan/utgafa/hlidarefni/J61-Johannessonetal.pdf). *Jökull* 61, 19–32. Source of the 2003 radar thickness figures and of the near-balance 1990s.
- Davíðsdóttir, Þ. (2003). Radio-echo sounding survey of Snæfellsjökull (cited in Jóhannesson et al., 2011).
- Hannesdóttir, H. et al. (2020). [A national glacier inventory and variations in glacier extent in Iceland from the Little Ice Age maximum to 2019](https://www.vedur.is/gogn/joklar/J70-1-34-minniupplausn.pdf). *Jökull* 70, 1–34. Balanced-budget ELAs of the monitored ice caps, and the source of the outlines.
- Farinotti, D. et al. (2019). [A consensus estimate for the ice thickness distribution of all glaciers on Earth](https://doi.org/10.3929/ethz-b-000315707). *Nature Geoscience* 12, 168–173. Modelled thickness maps for every glacier, including this one.
- Cuffey, K. M. and Paterson, W. S. B. (2010). *The Physics of Glaciers*, 4th ed. Perfect-plasticity thickness and accumulation area ratios.

## 10. Final outputs

A `final/` folder for exhibition prints, at higher resolution (`FINAL_DPI` for the sections, `FINAL_MAP_DPI` for the map):

- **Lines map:** hillshaded terrain, the area the glacier covered in 1999 or 2025 in black, the 1999 and 2025 outlines as dotted white lines, and the section lines in red. No labels.
- **Each line in `FINAL_LINES`:** its three figures (combined, 1990s, 2026) as drawn above, and the same three as `_ice_columns`: no locator map, nothing exaggerated, the ice in black, and grey columns rising straight up from every stretch of ice to the top edge of the image.

For `combined`, the `_ice_columns` figure is different: the 1990s and 2026 sections are stacked, 2026 on top, and the 1990s columns run up through the 2026 section to the top edge (`draw_stacked`).

In [ ]:
FINAL_LINES = ["ssw_nne_summit", "wnw_ese_north_flank"]
FINAL_DPI = 400        # sections
FINAL_MAP_DPI = 600    # lines map
LINE_COLOR = "#e10600" # section lines on the map
FINAL_DIR = OUT_DIR / "final"
FINAL_DIR.mkdir(parents=True, exist_ok=True)
for old in list(FINAL_DIR.glob("*.png")):
    old.unlink()

newest = PERIODS["2026"]
then_geom = outlines.loc[outlines["year"] == BALANCE_OUTLINE_YEAR, "geometry"].union_all()
now_geom = outlines.loc[outlines["year"] == newest, "geometry"].union_all()

# --- Lines map: hillshade, glacier extent in black, both outlines as dotted white lines, section lines in red
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(shade, cmap="gray", extent=extent, vmin=-0.3, vmax=1.3, interpolation="bilinear")
gpd.GeoSeries([then_geom.union(now_geom)], crs=crs).plot(ax=ax, color="black", linewidth=0)
for geom in (then_geom, now_geom):
    gpd.GeoSeries([geom], crs=crs).boundary.plot(ax=ax, color="white", linewidth=1.1, linestyle=(0, (1, 1.6)))
for name in FINAL_LINES:
    tr = transects[name]
    ax.plot([tr["x0"], tr["x1"]], [tr["y0"], tr["y1"]], color=LINE_COLOR, linewidth=2.2, solid_capstyle="butt")
ax.set_xlim(bounds.left, bounds.right)
ax.set_ylim(bounds.bottom, bounds.top)
ax.set_aspect("equal")
ax.axis("off")
fig.savefig(FINAL_DIR / f"{SLUG}_lines_map.png", dpi=FINAL_MAP_DPI, bbox_inches="tight", pad_inches=0)
plt.show()

# --- Both lines: the three figures as drawn above, and the same with grey columns rising above the ice
for name in FINAL_LINES:
    tr = transects[name]
    for suffix, key, lines in FIGURES:
        variant = suffix.lstrip("_") or "combined"
        draw_section(tr, tr["profiles"][key], lines, FINAL_DIR / f"{name}_{variant}.png", dpi=FINAL_DPI)
        if variant == "combined":  # the 1990s section with the 2026 one stacked on top
            draw_stacked(tr, tr["profiles"]["1990s"], tr["profiles"]["2026"], ["balance"], ["current"],
                         FINAL_DIR / f"{name}_{variant}_ice_columns.png", dpi=FINAL_DPI)
        else:
            draw_section(tr, tr["profiles"][key], lines, FINAL_DIR / f"{name}_{variant}_ice_columns.png",
                         show_map=False, columns=True, dpi=FINAL_DPI)

# --- README
rows = []
for name in FINAL_LINES:
    tr = transects[name]
    a, b = tr["ice_km"][BALANCE_OUTLINE_YEAR], tr["ice_km"][newest]
    rows.append(f"| `{name}` | {TRANSECTS[name][2]} | {a:.2f} → {b:.2f} km ({(b - a) / a:+.0%}) | [details](../{name}/README.md) |")
files = []
for name in FINAL_LINES:
    for variant, what in [("combined", f"{OUTLINE_YEAR} ice, both ELA lines"),
                          ("1990s", f"{BALANCE_OUTLINE_YEAR} ice, 1990 ELA"), ("2026", f"{newest} ice, 2026 ELA")]:
        files.append(f"| `{name}_{variant}.png` | {what}, with the locator map |")
        files.append(f"| `{name}_{variant}_ice_columns.png` | " + (
            f"the {BALANCE_OUTLINE_YEAR} section (1990 ELA) with the {newest} section (2026 ELA) stacked on top; "
            f"the {BALANCE_OUTLINE_YEAR} columns run up through the {newest} section to the top edge |"
            if variant == "combined" else f"{what}; grey columns rise from the ice to the top edge |"))
nf = transects["wnw_ese_north_flank"]
nf_change = abs((nf["ice_km"][newest] - nf["ice_km"][BALANCE_OUTLINE_YEAR]) / nf["ice_km"][BALANCE_OUTLINE_YEAR])
nf_top = max(np.nanmax(p["surface"]) for p in nf["profiles"].values())
(FINAL_DIR / "README.md").write_text(f"""# Snæfellsjökull cross-sections: final

![Map of the section lines]({SLUG}_lines_map.png)

`{SLUG}_lines_map.png` ({FINAL_MAP_DPI} dpi): hillshaded terrain; in black, the area the glacier covered in {BALANCE_OUTLINE_YEAR} or {newest}; dotted white lines, the {BALANCE_OUTLINE_YEAR} and {newest} outlines (the inner one is {newest}); in red, the two section lines. North is up; the map is {(bounds.right - bounds.left) / 1000:.1f} × {(bounds.top - bounds.bottom) / 1000:.1f} km.

| Line | Description | Ice along the line, {BALANCE_OUTLINE_YEAR} → {newest} | |
|---|---|---|---|
{chr(10).join(rows)}

## Figures ({FINAL_DPI} dpi)

| File | Shows |
|---|---|
{chr(10).join(files)}

All sections are at true scale (a metre up is as long as a metre across), with the ice in black on the rock mesh, seen from the side where the ground falls away and with the start of the line on the left. In the `_ice_columns` figures nothing is exaggerated: above every stretch of ice, a grey column rises straight up to the top edge of the image, marking where the glacier is along the line. A line's column figures share the same frame and rock, so they line up. In `_combined_ice_columns` the 1990s section sits at the bottom and the 2026 section directly on top of it; the 1990s columns carry on straight up through the 2026 section, and the columns are see-through, so where the 1990s and 2026 ice overlap the grey is darker. Where only the lighter 1990s column passes through the 2026 section, the ice has gone.

## The ELA lines on the north flank

The `wnw_ese_north_flank` line misses the summit and crosses only the north flank, which tops out at about {nf_top:,.0f} m. The 2026 ELA ({current['ela_m']:,.0f} m) is **above all of the ice on this line**, and the 1990 ELA ({balance['ela_m']:,.0f} m) is near its top. That's real, not an error: the ELA is one height for the whole glacier, and all of the ice on this line lies below it. This part of the glacier is in the zone that loses ice every summer and gains none, which is why the ice along this line shrank by {nf_change:.0%} since {BALANCE_OUTLINE_YEAR}. The ice that is left there survives because it is fed by ice flowing down from higher up and hasn't finished melting yet.

See `../README.md` and each line's folder for how the ice thickness and the ELA lines are estimated.
""", encoding="utf-8")
print("final:", sorted(p.name for p in FINAL_DIR.iterdir()))